In [1]:
import torch
import torch.nn as nn
import math
import psutil
import numpy as np
import pandas as pd
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import gc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [2]:
import math
import json
import torch
from torch.utils.data import Dataset

class TAMECriteoDataset(Dataset):
    def __init__(self, hf_dataset, dense_cols, cat_cols, dict_path):
        self.data = hf_dataset
        self.dense_cols = dense_cols
        self.cat_cols = cat_cols
        self.supported_dims = ["64", "32", "16", "8", "4", "2"]
        
        with open(dict_path, "r") as f:
            self.tame_dict = json.load(f)
            
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        
        # Xử lý Dense Features
        dense_x = []
        for c in self.dense_cols:
            val = row[c]
            if val is None or val < 0:
                dense_x.append(0.0)
            else:
                dense_x.append(math.log1p(float(val))) 
        
        # Xử lý Categorical Features (TAME Positional Routing)
        num_cats = len(self.cat_cols)
        grouped_cat = {dim: [0] * num_cats for dim in self.supported_dims}
        
        for col_idx, col in enumerate(self.cat_cols):
            val = row[col]
            if val is not None:
                gid = f"{col}_{val}"
                if gid in self.tame_dict:
                    info = self.tame_dict[gid]
                    dim_group = str(info["Dim_Group"])
                    local_idx = info["Local_Index"]
                    grouped_cat[dim_group][col_idx] = local_idx
        
        label = float(row['label'])

        return (
            torch.tensor(dense_x, dtype=torch.float32),
            {dim: torch.tensor(grouped_cat[dim], dtype=torch.long) for dim in self.supported_dims},
            torch.tensor(label, dtype=torch.float32)
        )

In [3]:
from datasets import load_dataset
from torch.utils.data import DataLoader

dense_cols = [f"I{i}" for i in range(1, 14)]
cat_cols = [f"C{i}" for i in range(1, 27)]

train_file = "/kaggle/input/datasets/nmpogg/criteo-cleaned-v2/train.parquet"
val_file = "/kaggle/input/datasets/nmpogg/criteo-cleaned-v2/validation.parquet"

print("Đang nạp dữ liệu từ các file Parquet...")
train_hf = load_dataset("parquet", data_files=train_file, split="train")
val_hf = load_dataset("parquet", data_files=val_file, split="train")

print(f"Train: {len(train_hf)} | Val: {len(val_hf)}")
print("Đang khởi tạo Dataset và DataLoader...")

dict_path = "/kaggle/input/datasets/nmpogg/criteo-cleaned-v2/tame_ivs_dictionary.json"
train_dataset = TAMECriteoDataset(train_hf, dense_cols, cat_cols, dict_path)
val_dataset = TAMECriteoDataset(val_hf, dense_cols, cat_cols, dict_path)

train_loader = DataLoader(
    train_dataset, 
    batch_size=8192, 
    shuffle=True, 
    num_workers=8,
    persistent_workers=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=8192, 
    shuffle=False, \
    num_workers=8, \
    persistent_workers=True, \
    pin_memory=True\
)
print("Hoàn tất DataLoader!")

Đang nạp dữ liệu từ các file Parquet...


Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/32 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Train: 36665662 | Val: 4583562
Đang khởi tạo Dataset và DataLoader...
Hoàn tất DataLoader!


In [4]:
import torch
import torch.nn as nn

class TAME_Embedding(nn.Module):
    def __init__(self, vocab_sizes, target_dim=64):
        super(TAME_Embedding, self).__init__()
        self.target_dim = target_dim
        self.supported_dims = [64, 32, 16, 8, 4, 2]
        self.embedding_blocks = nn.ModuleDict()
        
        for dim in self.supported_dims:
            dim_str = str(dim)
            if dim_str in vocab_sizes and vocab_sizes[dim_str] > 0:
                self.embedding_blocks[dim_str] = nn.Embedding(
                    num_embeddings=vocab_sizes[dim_str] + 1, 
                    embedding_dim=dim,
                    padding_idx=0
                )

    def forward(self, grouped_inputs):
        final_output = 0 
        for dim in self.supported_dims:
            dim_str = str(dim)
            if dim_str in grouped_inputs and dim_str in self.embedding_blocks:
                x = grouped_inputs[dim_str]
                emb = self.embedding_blocks[dim_str](x)
                
                if dim == self.target_dim:
                    out = emb
                else:
                    n_repeats = self.target_dim // dim
                    out = emb.repeat(1, 1, n_repeats) / n_repeats
                    
                final_output = final_output + out
        return final_output

In [5]:
import torch
import torch.nn as nn

class TAME_DeepFM(nn.Module):
    def __init__(self, vocab_sizes, num_dense_features, target_dim=64, k=13, hidden_dims=[256, 128, 64]):
        super(TAME_DeepFM, self).__init__()
        
        self.k = k
        self.target_dim = target_dim
        self.num_dense_features = num_dense_features
        self.num_cat_features = 26 
        self.num_total_features = num_dense_features + self.num_cat_features # 13 + 26 = 39 fields
        
        # TAME Embedding cho mảng thưa
        self.tame_emb = TAME_Embedding(vocab_sizes, target_dim=target_dim)
        
        # Nhúng cấu trúc biến liên tục lên không gian target_dim chung để thực hiện định tuyến
        self.dense_embs = nn.ModuleList([nn.Linear(1, target_dim) for _ in range(num_dense_features)])
        
        # Field Embedding (Giữ vững thuộc tính ngữ nghĩa vị trí của các trường dữ liệu)
        self.field_emb = nn.Embedding(self.num_total_features, target_dim)
        
        # Feature Router 2 tầng (Linear -> ReLU -> Linear) để sinh xác suất động cho 39 trường
        self.router = nn.Sequential(
            nn.Linear(target_dim, target_dim),
            nn.ReLU(),
            nn.Linear(target_dim, self.num_total_features)
        )
        
        # --- CẤU PHẦN ĐẦU RA DEEPFM (Xử lý trên k=13 đặc trưng được giữ lại) ---
        # 1. FM bậc 1 (Linear Component)
        self.fm_1st_layer = nn.Linear(target_dim, 1)
        
        # 2. Deep Component (MLP)
        deep_in_dim = self.k * target_dim # 13 * 64 = 832
        deep_layers = []
        in_dim = deep_in_dim
        for dim in hidden_dims:
            deep_layers.append(nn.Linear(in_dim, dim))
            deep_layers.append(nn.BatchNorm1d(dim)) 
            deep_layers.append(nn.ReLU())
            deep_layers.append(nn.Dropout(0.2))
            in_dim = dim
            
        deep_layers.append(nn.Linear(in_dim, 1))
        self.deep_net = nn.Sequential(*deep_layers)

    def forward(self, dense_x, grouped_cat_x):
        batch_size = dense_x.size(0)
        
        # 1. THU THẬP EMBEDDING TOÀN CỤC CỦA 39 TRƯỜNG
        # Biến đổi thưa qua TAME -> [Batch, 26, 64]
        cat_embs = self.tame_emb(grouped_cat_x)
        
        # Biến đổi liên tục qua Dense Layers -> [Batch, 13, 64]
        dense_embs_list = []
        for i in range(dense_x.size(1)):
            dense_embs_list.append(self.dense_embs[i](dense_x[:, i].unsqueeze(1)).unsqueeze(1))
        dense_embs = torch.cat(dense_embs_list, dim=1)
        
        # Hợp nhất thành một không gian tổng hợp -> [Batch, 39, 64]
        all_embs = torch.cat([dense_embs, cat_embs], dim=1)
        
        # Cộng hưởng với Field Embedding định danh cột đặc trưng
        field_indices = torch.arange(self.num_total_features, device=dense_x.device).unsqueeze(0).expand(batch_size, -1)
        all_embs = all_embs + self.field_emb(field_indices)
        
        # 2. DYNAMIC ROUTING CHỌN LỌC K=13 TRƯỜNG TỐI ƯU
        context = all_embs.mean(dim=1) # Pooling lấy ngữ cảnh đại diện [Batch, 64]
        router_probs = torch.softmax(self.router(context), dim=-1) # [Batch, 39]
        topk_probs, topk_indices = torch.topk(router_probs, self.k, dim=-1) # Lấy top k=13
        
        # Trích chọn các vector trường đồng thời kết hợp trọng số xác suất (Soft Routing)
        expanded_indices = topk_indices.unsqueeze(-1).expand(-1, -1, self.target_dim)
        selected_embs = torch.gather(all_embs, 1, expanded_indices) * topk_probs.unsqueeze(-1) # [Batch, 13, 64]
        
        # 3. THỰC THI MẠNG ĐẦU RA DEEPFM TRÊN KHÔNG GIAN K FEATURES
        # --- Cấu phần FM bậc 1 (Linear Component) ---
        # Biến đổi từng trường [Batch, 13, 64] -> [Batch, 13, 1], sau đó cộng tổng -> [Batch, 1]
        fm_1st = torch.sum(self.fm_1st_layer(selected_embs), dim=1)
            
        # --- Cấu phần FM bậc 2 (Feature Interaction) ---
        sum_of_emb = torch.sum(selected_embs, dim=1)         # [Batch, 64]
        sum_of_emb_sq = sum_of_emb ** 2                   # [Batch, 64]
        sq_of_emb = selected_embs ** 2                       # [Batch, 13, 64]
        sum_of_sq_emb = torch.sum(sq_of_emb, dim=1)       # [Batch, 64]
        
        fm_2nd = 0.5 * torch.sum(sum_of_emb_sq - sum_of_sq_emb, dim=1, keepdim=True) # [Batch, 1]
        
        # --- Cấu phần Deep MLP Component ---
        # Ép phẳng ma trận đặc trưng từ [Batch, 13, 64] sang vector đầu vào phẳng [Batch, 832]
        deep_in = selected_embs.view(selected_embs.size(0), -1) 
        deep_out = self.deep_net(deep_in) # [Batch, 1]
        
        # Kết hợp các thành phần điểm số
        out = fm_1st + fm_2nd + deep_out
        
        # Trả về logits và thêm topk_indices để đồng bộ lưu vết cho ô phân tích kiểm định
        return out.squeeze(1), topk_indices

In [6]:
import json

with open("/kaggle/input/datasets/nmpogg/criteo-cleaned-v2/tame_vocab_sizes.json", "r") as f:
    vocab_sizes = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()
print(f"Số GPU khả dụng: {num_gpus}")
for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

# Khởi tạo mô hình DeepFM tích hợp bộ chọn k=13 features quan trọng nhất
model = TAME_DeepFM(vocab_sizes=vocab_sizes, num_dense_features=len(dense_cols), k=13)

if num_gpus > 1:
    model = nn.DataParallel(model)

model = model.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)

Số GPU khả dụng: 1
  GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [7]:
import torch
from torch.utils.flop_counter import FlopCounterMode

print(f"{'Tên Component / Lớp (Layer)':<45} | {'Số lượng Params':<15}")
print("-" * 65)

total_params = 0

for name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue
    params = parameter.numel()
    clean_name = name.replace(".weight", "").replace(".bias", " (bias)")
    print(f"{clean_name:<45} | {params:,}")
    total_params += params

print("-" * 65)
print(f"{'TỔNG CỘNG THAM SỐ HUẤN LUYỆN:':<45} | {total_params:,}")
print("="*65)


print("Đang phân tích số lượng phép tính (FLOPs)...")
model.eval()

dummy_dense = torch.randn(1, len(dense_cols)).to(device)
dummy_cat = {
    dim: torch.zeros(1, len(cat_cols), dtype=torch.long).to(device) 
    for dim in ["64", "32", "16", "8", "4", "2"]
}

flop_counter = FlopCounterMode(model, display=True)
with flop_counter:
    _ = model(dummy_dense, dummy_cat)

Tên Component / Lớp (Layer)                   | Số lượng Params
-----------------------------------------------------------------
tame_emb.embedding_blocks.64                  | 554,179,776
tame_emb.embedding_blocks.32                  | 259,585,472
tame_emb.embedding_blocks.16                  | 131,691,008
tame_emb.embedding_blocks.8                   | 43,897,000
tame_emb.embedding_blocks.4                   | 8,779,404
tame_emb.embedding_blocks.2                   | 2,157,628
dense_embs.0                                  | 64
dense_embs.0 (bias)                           | 64
dense_embs.1                                  | 64
dense_embs.1 (bias)                           | 64
dense_embs.2                                  | 64
dense_embs.2 (bias)                           | 64
dense_embs.3                                  | 64
dense_embs.3 (bias)                           | 64
dense_embs.4                                  | 64
dense_embs.4 (bias)                           | 64
dense

/tmp/ipykernel_163/1367594755.py:31: UserWarning: mods argument is not needed anymore, you can stop passing it
  flop_counter = FlopCounterMode(model, display=True)


Module                         FLOP    % Total
-------------------------  --------  ---------
TAME_DeepFM                524.544K    100.00%
 - aten.addmm              524.544K    100.00%
 TAME_DeepFM.deep_net      508.032K     96.85%
  - aten.addmm             508.032K     96.85%
 TAME_DeepFM.fm_1st_layer    1.664K      0.32%
  - aten.addmm               1.664K      0.32%
 TAME_DeepFM.router         13.184K      2.51%
  - aten.addmm              13.184K      2.51%


In [7]:
from sklearn.metrics import roc_auc_score
from collections import Counter

print("Bắt đầu huấn luyện TAME_DeepFM...")
best_val_auc = 0.0 
save_path = "best_tame_deepfm_model.pth"
EPOCHS = 5

torch.cuda.reset_peak_memory_stats()
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0

    total_ram_gb = 0.0
    total_vram_gb = 0.0
    batch_count = 0
    
    train_targets, train_preds = [], []

    train_bar = tqdm(train_loader, desc=f"[TRAIN] Epoch {epoch+1}/{EPOCHS}")
    for dense_x, grouped_cat_x, labels in train_bar:
        dense_x = dense_x.to(device, non_blocking=True)
        labels  = labels.to(device, non_blocking=True)
        
        grouped_cat_x = {dim: t.to(device, non_blocking=True) for dim, t in grouped_cat_x.items()}

        optimizer.zero_grad()
        logits, _ = model(dense_x, grouped_cat_x) # Cập nhật unpack nhận thêm topk_indices
        loss = criterion(logits, labels)
        
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        probs = torch.sigmoid(logits)
        train_targets.extend(labels.detach().cpu().numpy())
        train_preds.extend(probs.detach().cpu().numpy())

        current_ram = psutil.virtual_memory().used / (1024**3)
        current_vram = torch.cuda.memory_allocated() / (1024**3)
        
        total_ram_gb += current_ram
        total_vram_gb += current_vram
        batch_count += 1
        
        train_bar.set_postfix({"Loss": f"{loss.item():.4f}"})

    avg_train_loss = train_loss / len(train_loader)
    train_auc = roc_auc_score(train_targets, train_preds)

    # val
    model.eval()
    val_loss = 0.0
    val_targets, val_preds = [], []

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"[VAL] Epoch {epoch+1}/{EPOCHS}", leave=False)
        for dense_x, grouped_cat_x, labels in val_bar:
            dense_x = dense_x.to(device, non_blocking=True)
            labels  = labels.to(device, non_blocking=True)
            grouped_cat_x = {dim: t.to(device, non_blocking=True) for dim, t in grouped_cat_x.items()}

            logits, _ = model(dense_x, grouped_cat_x) # Cập nhật unpack nhận thêm topk_indices
            loss = criterion(logits, labels)
            val_loss += loss.item()

            probs = torch.sigmoid(logits)
            val_targets.extend(labels.cpu().numpy())
            val_preds.extend(probs.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_auc = roc_auc_score(val_targets, val_preds)
    
    avg_ram = total_ram_gb / batch_count
    avg_vram = total_vram_gb / batch_count
    max_vram = torch.cuda.max_memory_allocated() / (1024**3)

    print(f"Tài nguyên Epoch {epoch+1}:")
    print(f"- RAM trung bình:  {avg_ram:.2f} GB")
    print(f"- VRAM trung bình: {avg_vram:.2f} GB")
    print(f"- VRAM đỉnh điểm:  {max_vram:.2f} GB")
    
    print(
        f"Epoch {epoch+1}: \n"
        f"Train Loss: {avg_train_loss:.4f} | Train AUC: {train_auc:.4f} || "
        f"Val Loss: {avg_val_loss:.4f} | Val AUC: {val_auc:.4f}\\n"
    )

    if val_auc > best_val_auc:
        print(f"Best Val AUC: {val_auc:.4f}. Đang lưu model...")
        best_val_auc = val_auc
        torch.save(model.state_dict(), save_path)
    else:
        print(f"Val AUC không cải thiện.")
    gc.collect()

Bắt đầu huấn luyện TAME_DeepFM...


[TRAIN] Epoch 1/5:   0%|          | 0/4476 [00:02<?, ?it/s]

[VAL] Epoch 1/5:   0%|          | 0/560 [00:00<?, ?it/s]

Tài nguyên Epoch 1:
- RAM trung bình:  63.68 GB
- VRAM trung bình: 14.94 GB
- VRAM đỉnh điểm:  18.67 GB
Epoch 1: 
Train Loss: 0.4316 | Train AUC: 0.8180 || Val Loss: 0.4083 | Val AUC: 0.8398\n
Best Val AUC: 0.8398. Đang lưu model...


[TRAIN] Epoch 2/5:   0%|          | 0/4476 [00:01<?, ?it/s]

...

[TRAIN] Epoch 3/5:   0%|          | 0/4476 [00:01<?, ?it/s]

[VAL] Epoch 3/5:   0%|          | 0/560 [00:00<?, ?it/s]

Tài nguyên Epoch 3:
- RAM trung bình:  86.92 GB
- VRAM trung bình: 14.94 GB
- VRAM đỉnh điểm:  18.67 GB
Epoch 3: 
Train Loss: 0.3845 | Train AUC: 0.8588 || Val Loss: 0.4527 | Val AUC: 0.8057\n
Val AUC không cải thiện.


[TRAIN] Epoch 4/5:   0%|          | 0/4476 [00:01<?, ?it/s]

KeyboardInterrupt: 

In [8]:
import torch
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
from collections import Counter

test_file = "/kaggle/input/datasets/nmpogg/criteo-cleaned-v2/test.parquet"
test_hf = load_dataset("parquet", data_files=test_file, split="train") # Đã loại bỏ hoàn toàn lỗi SyntaxError dấu gạch chéo ngược
test_dataset = TAMECriteoDataset(test_hf, dense_cols, cat_cols, "/kaggle/input/datasets/nmpogg/criteo-cleaned-v2/tame_ivs_dictionary.json")

test_loader = DataLoader(
    test_dataset, 
    batch_size=8192, 
    shuffle=False, 
    num_workers=8, 
    prefetch_factor=2,
    persistent_workers=True, 
    pin_memory=True
)


print("Bắt đầu đánh giá trên tập TEST, đo Latency & Phân tích Router...")

save_path = "best_tame_deepfm_model.pth"
state_dict = torch.load(save_path, map_location=device)
if next(iter(state_dict.keys())).startswith('module.'):
    state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}

model.load_state_dict(state_dict)
model.eval()

start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)

total_latency_ms = 0.0
total_samples = 0
latency_batch_count = 0
WARMUP_BATCHES = 5

test_loss = 0.0
test_targets, test_preds = [], []
all_combinations = []

with torch.no_grad():
    test_bar = tqdm(test_loader, desc="[TEST] Evaluating")
    for i, (dense_x, grouped_cat_x, labels) in enumerate(test_bar):
        
        dense_x = dense_x.to(device, non_blocking=True)
        labels  = labels.to(device, non_blocking=True)
        grouped_cat_x = {dim: t.to(device, non_blocking=True) for dim, t in grouped_cat_x.items()}

        batch_size = dense_x.size(0)

        start_event.record()
        logits, topk_indices = model(dense_x, grouped_cat_x)
        end_event.record()
        
        loss = criterion(logits, labels)
        test_loss += loss.item()

        probs = torch.sigmoid(logits)
        test_targets.extend(labels.cpu().numpy())
        test_preds.extend(probs.cpu().numpy())
        
        # Thu thập các tổ hợp index cột (0 -> 38) phục vụ thống kê Dynamic Router
        for row in topk_indices.cpu().numpy():
            all_combinations.append(tuple(sorted(row)))
        
        torch.cuda.synchronize()
        
        if i >= WARMUP_BATCHES:
            batch_latency = start_event.elapsed_time(end_event)
            total_latency_ms += batch_latency
            total_samples += batch_size
            latency_batch_count += 1

avg_test_loss = test_loss / len(test_loader)
test_auc = roc_auc_score(test_targets, test_preds)

if latency_batch_count > 0:
    avg_batch_latency = total_latency_ms / latency_batch_count
    avg_inference_per_sample = total_latency_ms / total_samples
else:
    avg_batch_latency = 0
    avg_inference_per_sample = 0

print(f"\nTest Loss: {avg_test_loss:.4f}")
print(f"Test AUC:  {test_auc:.4f}")

print(f"Latency 1 Batch (Size {test_loader.batch_size}): {avg_batch_latency:.2f} ms")
print(f"Latency 1 Dòng (Per Sample):      {avg_inference_per_sample:.4f} ms")

# Thống kê quyết định phân bổ động của tầng Router trên tập Test công khai
counts = Counter(all_combinations)
print(f"\n======== [KẾT QUẢ ROUTER DYNAMIC ANALYSIS] ========")
print(f"- Tổng số dòng kiểm tra: {len(all_combinations)}")
print(f"- Số lượng tổ hợp feature khác biệt được chọn: {len(counts)}")
print(f"- Top 3 tổ hợp phổ biến nhất (Feature Indices từ 0 đến 38):")
for idx, (comb, freq) in enumerate(counts.most_common(3)):
    print(f"  #{idx+1}: {comb} (Xuất hiện {freq} lần)")

Generating train split: 0 examples [00:00, ? examples/s]

Bắt đầu đánh giá trên tập TEST, đo Latency & Phân tích Router...


[TEST] Evaluating:   0%|          | 0/561 [00:01<?, ?it/s]


Test Loss: 0.4079
Test AUC:  0.8399
Latency 1 Batch (Size 8192): 2.19 ms
Latency 1 Dòng (Per Sample):      0.0003 ms

======== [KẾT QUẢ ROUTER DYNAMIC ANALYSIS] ========
- Tổng số dòng kiểm tra: 4591393
- Số lượng tổ hợp feature khác biệt được chọn: 25061
- Top 3 tổ hợp phổ biến nhất (Feature Indices từ 0 đến 38):
  #1: (np.int64(0), np.int64(3), np.int64(5), np.int64(9), np.int64(10), np.int64(11), np.int64(17), np.int64(25), np.int64(30), np.int64(34), np.int64(35), np.int64(37), np.int64(38)) (Xuất hiện 160207 lần)
  #2: (np.int64(3), np.int64(4), np.int64(5), np.int64(9), np.int64(10), np.int64(11), np.int64(13), np.int64(17), np.int64(25), np.int64(26), np.int64(35), np.int64(37), np.int64(38)) (Xuất hiện 126498 lần)
  #3: (np.int64(1), np.int64(3), np.int64(4), np.int64(5), np.int64(9), np.int64(10), np.int64(11), np.int64(13), np.int64(17), np.int64(25), np.int64(26), np.int64(35), np.int64(38)) (Xuất hiện 119210 lần)
